# 시계열 데이터의 결측치 탐색 및 시각화 

```python
import pandas as pd 
import numpy as np

train_building_org = pd.read_csv("building_mart87_power_org.csv")
train_building_org['date_time'] = pd.to_datetime(train_building_org['date_time'])

display(train_building_org.head())

display(f"{train_building_org['date_time'].min().date()} ~ {train_building_org['date_time'].max().date()}")
```

---

### Result

```
	date_time	temp	wind	hum	power
0	2022-06-01 00:00:00	13.6	0.7	52.0	412.02
1	2022-06-01 01:00:00	13.4	0.0	57.0	413.82
2	2022-06-01 02:00:00	13.3	0.2	57.0	405.36
3	2022-06-01 03:00:00	11.9	0.6	66.0	392.58
4	2022-06-01 04:00:00	14.4	0.7	51.0	416.70
'2022-06-01 ~ 2022-08-24'
```

## 결측치 처리(선형 보간 법)

- `interpolate()` : 결측치를 지정된 방법(method)에 따라 채움. method = `linear` (선형 보간법)을 사용 
- `bfill()`

```python
df = train_building_org.copy()

# 결측치 처리: 선형 보간법 사용
df['wind_Interpolated'] = df['wind'].interpolate(method='linear')
# 선형 보간법 : 결측지가 있는 지점의 앞뒤 값을 직선으로 연결하여 중간 값을 채워 넣는 방식
# 데이터가 일정하게 변할 때 효과적 
# 결측치 처리: 전후 값으로 채우기
df['wind_Fill'] = df['wind'].bfill()
#bfill() : 결측치를 바로 다음에 있는유효한 값으로 채운다. 
#데이터가 일관되게 이어지는 경우 단기적 결측이 발생할 때 유용

# 결과 출력
print(df[['wind', 'wind_Interpolated', 'wind_Fill']].head())
```

```
                     wind  wind_Interpolated  wind_Fill
date_time                                              
2022-06-01 00:00:00   0.7                0.7        0.7
2022-06-01 01:00:00   0.0                0.0        0.0
2022-06-01 02:00:00   0.2                0.2        0.2
2022-06-01 03:00:00   0.6                0.6        0.6
2022-06-01 04:00:00   0.7                0.7        0.7
```




## 시계열 데이터의 다운 샘플링

- 시계열 데이터를 분석할 때, 더 큰 시간 간격으로 데이터를 요약하는 다운 샘플링 
- 다운샘플링을 통해 데이터의 노이즈를 줄이고, 전반적인 트랜드를 파악 할 수 있다. 

```python
weekly_data = df.resample('W').mean()
daily_data = df.resample('D').mean()

# 결과 출력
print("Daily Data:\n", daily_data.head())
print("Weekly Data:\n", weekly_data.head())

print(f"Daily Data: {weekly_data.index.min()} ~ {weekly_data.index.max()} ({len(daily_data)})")
print(f"Weekly Data: {weekly_data.index.min()} ~ {weekly_data.index.max()} ({len(weekly_data)})")
```

```
Daily Data:
                  temp      wind        hum      power  wind_Interpolated  \
date_time                                                                  
2022-06-01  20.279167  1.695833  44.958333  1093.0500           1.695833   
2022-06-02  19.325000  1.412500  69.333333  1077.9375           1.412500   
2022-06-03  22.816667  1.133333  67.791667  1121.5350           1.133333   
2022-06-04  22.837500  0.812500  64.750000  1166.5725           0.812500   
2022-06-05  21.404167  0.908333  69.000000  1059.2475           0.908333   

            wind_Fill  
date_time              
2022-06-01   1.695833  
2022-06-02   1.412500  
2022-06-03   1.133333  
2022-06-04   0.812500  
2022-06-05   0.908333  
Weekly Data:
                  temp      wind        hum        power  wind_Interpolated  \
date_time                                                                    
2022-06-05  21.332500  1.192500  63.166667  1103.668500           1.192500   
2022-06-12  19.267262  1.035119  75.714286   999.043929           1.035119   
2022-06-19  19.833929  0.934524  84.666667  1097.423571           0.934524   
2022-06-26  24.004167  1.459524  85.279762  1110.493929           1.459524   
2022-07-03  25.369811  2.151786  90.937500  1269.092143           2.151786   

            wind_Fill  
date_time              
2022-06-05   1.192500  
2022-06-12   1.035119  
2022-06-19   0.934524  
2022-06-26   1.459524  
2022-07-03   2.151786  
Daily Data: 2022-06-05 00:00:00 ~ 2022-08-28 00:00:00 (85)
Weekly Data: 2022-06-05 00:00:00 ~ 2022-08-28 00:00:00 (13)
```

# Upsampling

```python
minutely_data = df.resample('10min').interpolate()
print(len(df))
print(len(minutely_data))
print(minutely_data.head())
print(minutely_data.index)
```